# 3D Baselines Comparison — GBM vs Others

Compare GBM-XZ against several 3D baselines on synthetic volume data.

In [ ]:
import numpy as np
import pandas as pd

# Try importing the newly ported 3D baselines
try:
    from gbm.core.mapper_3d import find_optimal_k_points_gbm_3D
    from gbm.baselines.greedy import find_optimal_k_points_greedy_3D
    from gbm.baselines.uniform import find_optimal_k_points_uniform_grid_search_3D
    print("3D GBM + Greedy + Uniform imported successfully")
except Exception as e:
    print(f"Import issue (some 3D code may still be wiring): {e}")

In [ ]:
# Tiny 3D synthetic field (same as previous notebook for consistency)
lw = 2
W, H, D = 100*lw, 100, 15
x = np.linspace(-20, 20, W)
y = np.linspace(0.4, 12, D)
z = np.linspace(-70, 70, H)
X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
C = 5e-6 + 3e-6 * np.exp(-((X)**2 + (Z)**2 + (Y-5)**2) / 60)

df = pd.DataFrame({"X": X.ravel(), "Y": Y.ravel(), "Z": Z.ravel(), "Carbon": C.ravel()})
inside = np.ones(W*H*D, dtype=bool)
gmean = float(C.mean())
print(f"3D synthetic volume ready, mean = {gmean:.2e}")

In [ ]:
# Run a couple of 3D methods (GBM + Greedy as example)
results = {}
try:
    loss_gbm, *_ = find_optimal_k_points_gbm_3D(df, inside, k=5, in_CO2_avg=gmean, cross_section="XZ", barn_LW_ratio=lw, epochs=5)
    results["GBM-XZ"] = loss_gbm
except Exception as e: results["GBM-XZ"] = f"error: {e}"

try:
    loss_greedy, *_ = find_optimal_k_points_greedy_3D(df, inside, k=5, in_CO2_avg=gmean, barn_LW_ratio=lw)
    results["Greedy-3D"] = loss_greedy
except Exception as e: results["Greedy-3D"] = f"error: {e}"

for name, val in results.items():
    print(f"{name:12s}: {val}")